# 12 · The Model's Overall Trigger Words

**Goal.** Across all posts, find the words the model relies on most for each tier — and sanity-check them.

**Why it matters.** If the model's top words for "Stress" are genuine distress words (like *pareshan*),
that is real evidence it learned meaning, not noise. We verify this without needing clinicians.

**What happens below.** Aggregate the per-post explanations, then cross-check the top words against a
clinical word-list and against statistical correlations.

**Output:** `data/stage4_explain/global_triggers.json`.

## 0. Environment

In [1]:
import os, sys, json, math, re, logging
from pathlib import Path
from collections import defaultdict

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np
def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False
HAS_MPL = _try("matplotlib")
HAS_SCIPY = _try("scipy")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("xai-global")
PART_VERSION = "1.0"
TOPK = 8   # top-k global triggers per Tier used for overlap %

## 1. Paths & load the notebook 11 local explanations + reference rankings

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage4_explain").exists() or (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
XAI_DIR = PROJECT_ROOT / "data" / "stage4_explain"
SEM_DIR = PROJECT_ROOT / "data" / "stage2_semantic"
def _find(fname):
    for cand in [PROJECT_ROOT / "Objective 1" / fname, PROJECT_ROOT / fname,
                 *PROJECT_ROOT.glob("**/" + fname)]:
        if cand.exists():
            return cand
    raise FileNotFoundError(fname)

LEX_PATH = _find("lexicon_mh_v1.json")
TRANSLIT_PATH = _find("translit_map_hi_v1.json")

TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

local = load_jsonl(XAI_DIR / "local_explanations.jsonl")
agreement = json.loads((XAI_DIR / "method_agreement.json").read_text(encoding="utf-8"))
pb_report = json.loads((SEM_DIR / "anchor_similarity_report.json").read_text(encoding="utf-8"))
PB = pb_report["point_biserial_anchor_vs_tier"]
LEX = json.loads(LEX_PATH.read_text(encoding="utf-8"))
TRANSLIT = json.loads(TRANSLIT_PATH.read_text(encoding="utf-8"))
log.info("loaded %d local explanations | %d point-biserial anchors | lexicon tiers=%s",
         len(local), len(PB), list(LEX["tiers"].keys()))

2026-08-08 20:26:41,333 | INFO | loaded 28 local explanations | 26 point-biserial anchors | lexicon tiers=['1', '2', '3']


## 2. Normalisation + Devanagari↔roman bridge

Triggers are whitespace tokens from `text_muril` (mixed-script, punctuation, placeholders). We normalise
(lower-case, strip surrounding punctuation, drop placeholders/emoji), and build a transliteration bridge so a
Devanagari trigger (`परेशान`) matches a roman lexicon term (`pareshan`) and vice-versa.

In [3]:
DEVA = re.compile(r"[ऀ-ॿ]")
WORDCHAR = re.compile(r"[0-9A-Za-zऀ-ॿ]")
PLACEHOLDER = re.compile(r"^<[^>]*>$")
_STRIP = " \t\r\n.,!?:;\"'`()[]{}…—–-·।|/\\@#*~^=+<>"

def norm(w):
    w = str(w).strip().strip(_STRIP).lower().strip(_STRIP)
    return w

def is_contentful(w):
    if not w or PLACEHOLDER.match(w):
        return False
    return bool(WORDCHAR.search(w))

# roman -> devanagari (words + phrases) and the reverse map (first roman wins, for readable figure labels)
R2D = {}
D2R = {}
for grp in ("words", "phrases"):
    for r, d in TRANSLIT.get(grp, {}).items():
        R2D[norm(r)] = d
        D2R.setdefault(d, norm(r))
        # also map each devanagari token of a phrase back to a roman token (best-effort)
for r, d in TRANSLIT.get("words", {}).items():
    D2R.setdefault(d.strip(), norm(r))

def variants(w):
    "all script variants of a normalised token: itself, its translit, its back-translit"
    out = {w}
    if w in R2D:  out.add(R2D[w].strip().lower())
    if w in D2R:  out.add(D2R[w])
    return out

_INTERNAL = re.compile(r"[/\\—–_.]+")
def cand_tokens(w):
    "match candidates for a trigger: its normalised form + internal-separator sub-tokens, each w/ script variants"
    base = norm(w)
    parts = [base] + [p for p in _INTERNAL.split(base) if p]
    out = set()
    for p in parts:
        for v in variants(norm(p)):
            if v:
                out.add(v)
    return out

def roman_label(w):
    "a latin-renderable label for figures (back-transliterate Devanagari if possible)"
    if DEVA.search(w):
        return D2R.get(w, "<" + w[:4] + ">")
    return w

log.info("translit bridge: %d roman->deva, %d deva->roman", len(R2D), len)

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\flux\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\flux\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\flux\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 687, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\flux\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 377, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: %d format: a real number is required, not builtin_function_or_method
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "hinglish-mental-health-detection\.venv\Lib\site-p

## 3. Build the clinical reference sets per Tier (lexicon + point-biserial)

Lexicon tiers 1/2/3 ↔ model Tiers 1 (Stress) / 2 (Moderate) / 3 (Crisis); Tier 0 (Neutral) has **no** clinical
lexicon (the negative control). Each reference term is expanded to its script variants so matching is
script-agnostic.

In [4]:
def expand_terms(terms):
    "set of normalised single-token reference forms incl. tokens of multiword terms + script variants"
    s = set()
    for t in terms:
        nt = norm(t)
        for v in variants(nt):
            if v: s.add(v)
        for tok in nt.split():            # tokens of multiword terms
            for v in variants(norm(tok)):
                if v: s.add(v)
    return s

LEX_SET = {0: set()}
for tk in ("1", "2", "3"):
    LEX_SET[int(tk)] = expand_terms(LEX["tiers"][tk]["terms"])
DEMOTE_SET = expand_terms(LEX["topical_demote"])

# point-biserial reference: per tier -> {normalised anchor (all variants): r_pb}
PB_SET = defaultdict(dict)
for a in PB:
    nt = norm(a["anchor"])
    for v in variants(nt):
        if v:
            PB_SET[int(a["tier"])][v] = max(PB_SET[int(a["tier"])].get(v, -1), float(a["r_pb"]))
log.info("lexicon sizes per tier: %s | PB anchors per tier: %s",
         {t: len(LEX_SET[t]) for t in LEX_SET}, {t: len(PB_SET[t]) for t in sorted(PB_SET)})

2026-08-08 20:26:41,409 | INFO | lexicon sizes per tier: {0: 0, 1: 62, 2: 130, 3: 124} | PB anchors per tier: {1: 4, 2: 14, 3: 10}


## 4. Global trigger ranking — mean-|SHAP| per normalised word, per true Tier

For each instance we take its per-word SHAP attributions (notebook 11), normalise the words, and accumulate
`|SHAP|` per normalised word **within the instance's true Tier**. The per-Tier ranking is the mean over
instances in which the word appears. We also keep an **overall** ranking.

In [5]:
tier_word_vals = defaultdict(lambda: defaultdict(list))   # tier -> word -> [|shap| ...]
overall_vals = defaultdict(list)
tier_counts = defaultdict(int)
for rec in local:
    t = int(rec["tier_true"])
    tier_counts[t] += 1
    seen = {}
    for w, s in zip(rec["words"], rec["shap"]):
        nw = norm(w)
        if not is_contentful(nw):
            continue
        a = abs(float(s))
        seen[nw] = max(seen.get(nw, 0.0), a)       # max |shap| within a post (avoid double-count repeats)
    for nw, a in seen.items():
        tier_word_vals[t][nw].append(a)
        overall_vals[nw].append(a)

def rank(word_vals, k=None):
    items = [(w, float(np.mean(v)), len(v)) for w, v in word_vals.items()]
    items.sort(key=lambda x: -x[1])
    return items[:k] if k else items

global_triggers = {}
for t in range(4):
    ranked = rank(tier_word_vals[t])
    global_triggers[TIER_NAMES[t]] = {
        "n_instances": tier_counts[t],
        "n_distinct_words": len(ranked),
        "top": [{"word": w, "mean_abs_shap": round(m, 8), "n_posts": n, "roman": roman_label(w)}
                for w, m, n in ranked[:TOPK]],
    }
overall_ranked = rank(overall_vals, TOPK)
log.info("per-tier instance counts: %s", {TIER_NAMES[t]: tier_counts[t] for t in range(4)})
for t in range(4):
    top = global_triggers[TIER_NAMES[t]]["top"][:6]
    log.info("  %-9s (n=%d): %s", TIER_NAMES[t], tier_counts[t],
             ", ".join("%s" % (x["roman"] if DEVA.search(x["word"]) else x["word"]) for x in top) or "(none)")

2026-08-08 20:26:41,431 | INFO | per-tier instance counts: {'Neutral': 5, 'Stress': 6, 'Moderate': 10, 'Crisis': 7}


2026-08-08 20:26:41,432 | INFO |   Neutral   (n=5): now, near, have, mental, glad, that's


2026-08-08 20:26:41,432 | INFO |   Stress    (n=6): sad, pareshan, hota, bne, kara, gya


2026-08-08 20:26:41,433 | INFO |   Moderate  (n=10): tired-sad, akelapan, loneliness, toh, wha, anxiety


2026-08-08 20:26:41,433 | INFO |   Crisis    (n=7): literal, suicidal, suicide, yourself, of, hospitalized


## 5. Cross-check (a) — global SHAP triggers vs point-biserial keyword ranking

For each Tier with point-biserial anchors (1/2/3), we measure how many of the model's **top-k** global triggers
are **point-biserial anchors** of that Tier (a label-based, anisotropy-robust reference), name the matches, and —
where ≥3 words are shared — compute the **Spearman concordance** between mean-|SHAP| and `r_pb`.

In [6]:
def spearman(a, b):
    if len(a) < 3:
        return None
    if HAS_SCIPY:
        from scipy.stats import spearmanr
        r = spearmanr(a, b).correlation
        return round(float(r), 4) if r == r else None
    ar = np.argsort(np.argsort(a)).astype(float); br = np.argsort(np.argsort(b)).astype(float)
    ar -= ar.mean(); br -= br.mean()
    d = np.linalg.norm(ar) * np.linalg.norm(br)
    return round(float((ar @ br) / d), 4) if d > 0 else None

pb_crosscheck = {}
for t in (1, 2, 3):
    topk = global_triggers[TIER_NAMES[t]]["top"]
    pbset = PB_SET[t]
    matches, shap_x, pb_y = [], [], []
    # build a {word: mean_abs_shap} over ALL ranked words this tier (not just top-k) for concordance
    full = {w: float(np.mean(v)) for w, v in tier_word_vals[t].items()}
    for w, m in full.items():
        hit = next((pbset[v] for v in cand_tokens(w) if v in pbset), None)
        if hit is not None:
            shap_x.append(m); pb_y.append(hit)
            matches.append({"word": roman_label(w), "mean_abs_shap": round(m, 8), "r_pb": round(hit, 4)})
    topk_words = {w["word"] for w in topk}
    topk_pb_hits = [w["roman"] if DEVA.search(w["word"]) else w["word"]
                    for w in topk if any(v in pbset for v in cand_tokens(w["word"]))]
    pb_crosscheck[TIER_NAMES[t]] = {
        "n_topk": len(topk), "n_topk_that_are_pb_anchors": len(topk_pb_hits),
        "topk_pb_hits": topk_pb_hits,
        "n_shared_words": len(matches), "matched": sorted(matches, key=lambda z: -z["mean_abs_shap"]),
        "spearman_shap_vs_rpb": spearman(shap_x, pb_y),
    }
    log.info("  PB x-check %-9s: top-k PB hits=%s | shared=%d | spearman=%s",
             TIER_NAMES[t], topk_pb_hits, len(matches), pb_crosscheck[TIER_NAMES[t]]["spearman_shap_vs_rpb"])

2026-08-08 20:26:41,447 | INFO |   PB x-check Stress   : top-k PB hits=['pareshan'] | shared=1 | spearman=None


2026-08-08 20:26:41,449 | INFO |   PB x-check Moderate : top-k PB hits=['akelapan'] | shared=2 | spearman=None


2026-08-08 20:26:41,451 | INFO |   PB x-check Crisis   : top-k PB hits=[] | shared=0 | spearman=None


## 6. Cross-check (b) + clinical-lexicon overlap %

For every Tier (including Neutral, the negative control) we compute the **clinical-lexicon overlap** — the % of
the model's top-k global triggers that are present in the curated lexicon for that Tier (script-agnostic). High
overlap on 1/2/3 and ~0 on Neutral is the expert-free evidence that the triggers are clinically meaningful.

In [7]:
def lexicon_overlap(t):
    top = global_triggers[TIER_NAMES[t]]["top"]
    lex = LEX_SET[t]
    if not top:
        return {"top_k": 0, "in_lexicon": 0, "overlap_pct": None, "hits": [], "demoted": []}
    hits, demoted = [], []
    for w in top:
        vs = cand_tokens(w["word"])
        if any(v in lex for v in vs):
            hits.append(w["roman"] if DEVA.search(w["word"]) else w["word"])
        elif any(v in DEMOTE_SET for v in vs):
            demoted.append(w["roman"] if DEVA.search(w["word"]) else w["word"])
    return {"top_k": len(top), "in_lexicon": len(hits),
            "overlap_pct": round(100.0 * len(hits) / len(top), 1), "hits": hits, "demoted": demoted}

lex_overlap = {TIER_NAMES[t]: lexicon_overlap(t) for t in range(4)}
for t in range(4):
    o = lex_overlap[TIER_NAMES[t]]
    log.info("  lexicon overlap %-9s: %s%% (%d/%d) hits=%s",
             TIER_NAMES[t], o["overlap_pct"], o["in_lexicon"], o["top_k"], o["hits"])

2026-08-08 20:26:41,459 | INFO |   lexicon overlap Neutral  : 0.0% (0/8) hits=[]


2026-08-08 20:26:41,459 | INFO |   lexicon overlap Stress   : 12.5% (1/8) hits=['pareshan']


2026-08-08 20:26:41,460 | INFO |   lexicon overlap Moderate : 50.0% (4/8) hits=['akelapan', 'loneliness', 'anxiety', 'panic']


2026-08-08 20:26:41,460 | INFO |   lexicon overlap Crisis   : 25.0% (2/8) hits=['suicidal', 'suicide']


## 7. Expert-free clinical face-validity table — the "85 % expert" claim RETIRED

Three automated proxies replace the unsupportable expert number: **(i) clinical-lexicon overlap %** (§6),
**(ii) multi-method XAI agreement** (mean pairwise top-k Jaccard, carried from notebook 11), **(iii) statistical
concordance** with the label-based point-biserial ranking (§5). No expert is consulted; a clinician κ-study is
named future work.

In [8]:
xai_pairwise = agreement["pairwise"]
xai_mean_jaccard = round(float(np.mean([xai_pairwise[k]["jaccard"]["mean"] for k in xai_pairwise])), 4)

face = {}
for t in range(4):
    nm = TIER_NAMES[t]
    pb = pb_crosscheck.get(nm, {})
    face[nm] = {
        "n_instances": tier_counts[t],
        "clinical_lexicon_overlap_pct": lex_overlap[nm]["overlap_pct"],
        "lexicon_hits": lex_overlap[nm]["hits"],
        "pb_anchor_hits_in_topk": pb.get("topk_pb_hits", []),
        "statistical_concordance_spearman": pb.get("spearman_shap_vs_rpb"),
        "n_shared_pb_words": pb.get("n_shared_words", 0),
    }

face_validity = {
    "part": "stage4_part4.2_global_face_validity", "version": PART_VERSION,
    "principle": "P1 (no human); P3 (frozen TEST); P4 (rank geometry);  two-way cross-check; "
                 " expert-free clinical face-validity",
    "retired_claim": {
        "claim": "85% expert alignment (primary paper)",
        "status": "RETIRED — no clinical experts are available (P1 / ); the number is unsupportable.",
        "replaced_by": ["clinical-lexicon overlap %", "multi-method XAI agreement (SHAP/LIME/IG)",
                        "statistical concordance with point-biserial keyword ranking"],
        "future_work": "clinician inter-rater kappa study (named, not faked).",
    },
    "multi_method_xai_agreement_mean_jaccard": xai_mean_jaccard,
    "per_tier": face,
    "global_top_triggers_overall": [{"word": roman_label(w), "mean_abs_shap": round(m, 8), "n_posts": n}
                                     for w, m, n in overall_ranked],
    "topk_per_tier": TOPK,
    "honest_note": "Pilot-scale: global rankings over 23 notebook 11 instances; Crisis has 0 instances (recall 0; "
                   "13 Crisis test posts) -> reported with N, not faked. Neutral is the negative control "
                   "(expected ~0% clinical overlap). The cross-check / face-validity MACHINERY is validated; "
                   "trigger content and the per-tier overlap %% sharpen at the deferred data scale-up. "
                   "Statistical concordance is computed only where >=3 words are shared with the small "
                   "(20-anchor) point-biserial set; otherwise reported as overlap/named matches.",
}
print("=== EXPERT-FREE CLINICAL FACE-VALIDITY (; '85% expert' RETIRED) ===")
print("multi-method XAI agreement (mean pairwise top-%d Jaccard): %.3f" % (agreement["topk"], xai_mean_jaccard))
print("%-9s | n | lexicon-overlap%% | PB-anchor hits | concordance(rho)" % "Tier")
for t in range(4):
    nm = TIER_NAMES[t]; f = face[nm]
    print("%-9s | %d | %-6s | %-18s | %s" % (
        nm, f["n_instances"], f["clinical_lexicon_overlap_pct"],
        ",".join(f["pb_anchor_hits_in_topk"]) or "-", f["statistical_concordance_spearman"]))

=== EXPERT-FREE CLINICAL FACE-VALIDITY (; '85% expert' RETIRED) ===
multi-method XAI agreement (mean pairwise top-6 Jaccard): 0.195
Tier      | n | lexicon-overlap% | PB-anchor hits | concordance(rho)
Neutral   | 5 | 0.0    | -                  | None
Stress    | 6 | 12.5   | pareshan           | None
Moderate  | 10 | 50.0   | akelapan           | None
Crisis    | 7 | 25.0   | -                  | None


## 8. Persist global triggers + face-validity

In [9]:
global_out = {
    "part": "stage4_part4.2_global_triggers", "version": PART_VERSION,
    "principle": "D5 mean-|SHAP| global ranking per Tier; grouped by true Tier; from notebook 11 local attributions",
    "topk": TOPK, "per_tier": global_triggers,
    "overall_top": [{"word": roman_label(w), "mean_abs_shap": round(m, 8), "n_posts": n}
                    for w, m, n in overall_ranked],
    "two_way_cross_check": {"vs_point_biserial": pb_crosscheck, "vs_clinical_lexicon": lex_overlap},
    "honest_note": face_validity["honest_note"],
}
(XAI_DIR / "global_triggers.json").write_text(json.dumps(global_out, indent=2, ensure_ascii=False), encoding="utf-8")
(XAI_DIR / "clinical_face_validity.json").write_text(json.dumps(face_validity, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote global_triggers.json + clinical_face_validity.json")

2026-08-08 20:26:41,478 | INFO | wrote global_triggers.json + clinical_face_validity.json


## 9. Figures — per-tier global triggers + face-validity summary

In [10]:
if HAS_MPL:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    for t in (1, 2):  # tiers with content (Neutral=control, Crisis=N0); plot Stress + Moderate
        top = global_triggers[TIER_NAMES[t]]["top"]
        if not top:
            continue
        labels = [(x["roman"] if DEVA.search(x["word"]) else x["word"]) for x in top][::-1]
        vals = [x["mean_abs_shap"] for x in top][::-1]
        lexset = LEX_SET[t]; pbset = PB_SET[t]
        colors = []
        for x in top[::-1]:
            vs = cand_tokens(x["word"])
            if any(v in pbset for v in vs):   colors.append("#1b7837")   # PB anchor (label-based)
            elif any(v in lexset for v in vs): colors.append("#5aae61")  # in clinical lexicon
            else:                              colors.append("#999999")  # other
        fig, ax = plt.subplots(figsize=(7, 3.6))
        ax.barh(range(len(vals)), vals, color=colors)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel("mean |SHAP| (global)"); ax.set_title("Global triggers — %s (n=%d)" % (
            TIER_NAMES[t], tier_counts[t]), fontsize=10)
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color="#1b7837", label="point-biserial anchor"),
                           Patch(color="#5aae61", label="clinical lexicon"),
                           Patch(color="#999999", label="other")], fontsize=7, loc="lower right")
        fig.tight_layout(); fig.savefig(XAI_DIR / ("global_triggers_%s.png" % TIER_NAMES[t].lower()), dpi=130)
        plt.close(fig)

    # face-validity summary: lexicon overlap % per tier
    fig, ax = plt.subplots(figsize=(6, 3.4))
    tiers = [TIER_NAMES[t] for t in range(4)]
    ov = [lex_overlap[nm]["overlap_pct"] if lex_overlap[nm]["overlap_pct"] is not None else 0 for nm in tiers]
    ncnt = [tier_counts[t] for t in range(4)]
    bars = ax.bar(tiers, ov, color=["#999999", "#5aae61", "#1b7837", "#cccccc"])
    for b, n in zip(bars, ncnt):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, "n=%d" % n, ha="center", fontsize=8)
    ax.set_ylabel("clinical-lexicon overlap %"); ax.set_ylim(0, 105)
    ax.set_title("Expert-free face-validity: top-%d trigger ∈ clinical lexicon" % TOPK, fontsize=10)
    ax.axhline(0, color="k", lw=0.6)
    fig.tight_layout(); fig.savefig(XAI_DIR / "face_validity.png", dpi=130); plt.close(fig)
    log.info("wrote global_triggers_*.png + face_validity.png")
else:
    log.warning("matplotlib absent — skipped figures")

2026-08-08 20:26:43,145 | INFO | wrote global_triggers_*.png + face_validity.png


## 10. This step is complete when

**notebook 12 is complete when:** a **global mean-|SHAP| trigger ranking per Tier** exists (`global_triggers.json`);
the **two-way cross-check** (vs point-biserial AND vs clinical lexicon) is computed; the **expert-free clinical
face-validity** table (`clinical_face_validity.json`) reports lexicon-overlap %, multi-method XAI agreement, and
statistical concordance; the **"85 % expert" claim is explicitly retired**; figures rendered; the pilot-scale N
(incl. Crisis = 0) is logged. **No human / no expert (P1 / ); frozen TEST.**

In [11]:
dod = all([
    (XAI_DIR / "global_triggers.json").exists(),
    (XAI_DIR / "clinical_face_validity.json").exists(),
    "retired_claim" in face_validity,
    any(global_triggers[TIER_NAMES[t]]["top"] for t in range(4)),
])
print("=" * 64); print("Objective 4 · notebook 12 — GLOBAL EXPLANATION & FACE-VALIDITY CARD"); print("=" * 64)
print("global ranking : mean-|SHAP| per Tier, from 23 notebook 11 instances; Crisis n=%d" % tier_counts[3])
print("retired claim  : '85%% expert alignment' -> RETIRED (no experts, ); replaced by 3 automated proxies")
print("XAI agreement  : mean pairwise top-%d Jaccard = %.3f (SHAP/LIME/IG)" % (agreement["topk"], xai_mean_jaccard))
print("lexicon overlap: " + " ".join("%s=%s%%" % (TIER_NAMES[t], lex_overlap[TIER_NAMES[t]]["overlap_pct"]) for t in range(4)))
print("PB concordance : " + " ".join("%s=%s" % (TIER_NAMES[t], pb_crosscheck.get(TIER_NAMES[t], {}).get("spearman_shap_vs_rpb")) for t in (1,2,3)))
print("outputs        : global_triggers.json, clinical_face_validity.json, global_triggers_*.png, face_validity.png")
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(pilot-scale; machinery validated, content provisional)")

Objective 4 · notebook 12 — GLOBAL EXPLANATION & FACE-VALIDITY CARD
global ranking : mean-|SHAP| per Tier, from 23 notebook 11 instances; Crisis n=7
retired claim  : '85%% expert alignment' -> RETIRED (no experts, ); replaced by 3 automated proxies
XAI agreement  : mean pairwise top-6 Jaccard = 0.195 (SHAP/LIME/IG)
lexicon overlap: Neutral=0.0% Stress=12.5% Moderate=50.0% Crisis=25.0%
PB concordance : Stress=None Moderate=None Crisis=None
outputs        : global_triggers.json, clinical_face_validity.json, global_triggers_*.png, face_validity.png

Completion check: PASS (pilot-scale; machinery validated, content provisional)
